In [16]:
%%capture
!pip install unsloth
!pip install --no-deps --upgrade trl peft accelerate bitsandbytes
# If imports fail after install: Runtime > Restart session, then run all cells again.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Fix dependency conflict between Unsloth and Transformers
!pip uninstall -y unsloth transformers trl peft accelerate bitsandbytes
!pip install --force-reinstall --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l783ts38/unsloth_bbd6af8d252d4720ae78219041a58576
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l783ts38/unsloth_bbd6af8d252d4720ae78219041a58576
  Resolved https://github.com/unslothai/unsloth.git to commit 13ef43494732fbec65b1263ed4a8d0cb4b51a98c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 13.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 17.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 30.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 114.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 143.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [1]:
import sys
import re

filepath = "/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py"
try:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Modify torchvision_compatibility_check to return immediately
    old_func = "def torchvision_compatibility_check():"
    new_func = "def torchvision_compatibility_check():\n    return"

    if old_func in content and new_func not in content:
        content = content.replace(old_func, new_func)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        print("🎉 Successfully patched on-disk torchvision_compatibility_check function!")
    else:
        print("ℹ️ Function already patched or not found.")
except Exception as e:
    print(f"⚠️ Error patching file: {e}")

# Force reload clean slate for unsloth
for module in list(sys.modules.keys()):
    if "unsloth" in module:
        del sys.modules[module]

try:
    from unsloth import FastLanguageModel
    print("✅ Unsloth imported successfully! Ready to train.")
except Exception as e:
    print(f"❌ Import failed: {e}")

🎉 Successfully patched on-disk torchvision_compatibility_check function!


/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:115: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Detected that PyTorch and TorchAudio were compiled with different CUDA versions. PyTorch has CUDA version 13.2 whereas TorchAudio has CUDA version 13.0. Please install the TorchAudio version that matches your PyTorch version.
  disable_torchaudio_if_cuda_mismatched()
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 09:44:15.676000 6536 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 09:44:15.809000 6536 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth imported successfully! Ready to train.


In [2]:
import os, glob, zipfile, shutil

# 1) find your upload on Drive (top level + one folder deep)
cands = []
for pat in ('*training*', '*assessment*', '*candidate*', '*nimbus*'):
    cands += glob.glob(f'/content/drive/MyDrive/{pat}')
    cands += glob.glob(f'/content/drive/MyDrive/*/{pat}')
cands = sorted(set(cands))
print("Upload candidates:", cands)

# 2) unzip any zip into /content/pack
os.makedirs('/content/pack', exist_ok=True)
for c in cands:
    if c.lower().endswith('.zip'):
        with zipfile.ZipFile(c) as z:
            z.extractall('/content/pack')
        print("Extracted:", c)

# 3) search for the actual files (handles nested folders AND .txt renames)
targets = {'train': ['train.jsonl','train.txt'],
           'dev':   ['dev.jsonl','dev.txt'],
           'test':  ['test_inputs.jsonl','test_inputs.txt'],
           'score': ['score.py'],
           'schema':['SCHEMA.md','schema.md','schema.txt']}
located = {}
for base in ['/content/pack'] + [c for c in cands if os.path.isdir(c)]:
    for root, dirs, files in os.walk(base):
        for key, names in targets.items():
            if key not in located:
                for n in names:
                    if n in files:
                        located[key] = os.path.join(root, n)
print("Located:", located)

# 4) copy into the standard layout: /content/pack/data/*.jsonl + score.py
os.makedirs('/content/pack/data', exist_ok=True)
for key, fname in [('train','train.jsonl'), ('dev','dev.jsonl'), ('test','test_inputs.jsonl')]:
    if key in located:
        shutil.copy(located[key], f'/content/pack/data/{fname}')
if 'score'  in located: shutil.copy(located['score'],  '/content/pack/score.py')
if 'schema' in located: shutil.copy(located['schema'], '/content/pack/SCHEMA.md')

PACK = '/content/pack'
print("PACK =", PACK)
print("contents:", os.listdir(PACK))
print("data:", os.listdir(PACK + '/data'))

# if we couldn't find the files, show what IS on your Drive so we can locate them
if len(located) < 3:
    print("MY DRIVE TOP LEVEL:", os.listdir('/content/drive/MyDrive'))

Upload candidates: ['/content/drive/MyDrive/Colab Notebooks/nimbuspay.ipynb', '/content/drive/MyDrive/nimbuspay', '/content/drive/MyDrive/nimbuspay_3b', '/content/drive/MyDrive/training_assessment', '/content/drive/MyDrive/training_assessment/candidate_pack', '/content/drive/MyDrive/training_assessment/nimbuspay.ipynb']
Located: {'score': '/content/drive/MyDrive/training_assessment/candidate_pack/score.py', 'schema': '/content/drive/MyDrive/training_assessment/candidate_pack/SCHEMA.md', 'train': '/content/drive/MyDrive/training_assessment/candidate_pack/data/train.jsonl', 'dev': '/content/drive/MyDrive/training_assessment/candidate_pack/data/dev.jsonl', 'test': '/content/drive/MyDrive/training_assessment/candidate_pack/data/test_inputs.jsonl'}
PACK = /content/pack
contents: ['data', 'SCHEMA.md', 'score.py']
data: ['dev.jsonl', 'train.jsonl', 'test_inputs.jsonl']


In [3]:
import os, json, re, csv
from datetime import date, timedelta

SAVE = "/content/drive/MyDrive/nimbuspay"; os.makedirs(SAVE, exist_ok=True)
PACK = "/content/pack"
def load(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

train = load(f'{PACK}/data/train.jsonl')
dev   = load(f'{PACK}/data/dev.jsonl')
test  = load(f'{PACK}/data/test_inputs.jsonl')
print("rows:", len(train), len(dev), len(test))   # expect: 1445 200 400

src = open(f'{PACK}/score.py', encoding='utf-8').read()
print("scorer OK" if '"category"' in src else "WARNING: scorer looks corrupted")

rows: 1445 200 400
scorer OK


In [4]:
import json, re, csv
from datetime import date, timedelta

KEYS = ["category","priority","amount_paise","txn_date","txn_id","channel","language","needs_human"]
CATS = {"payment_failed","refund","fraud","account_access","kyc","offers","other"}
CHANS = {"upi","card","netbanking","wallet"}
NULL_CATS = {"account_access","kyc","other"}

LEGAL  = re.compile(r"\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action", re.I)
REPEAT = re.compile(r"third time|second time|second complaint|third complaint|again and again|baar baar|"
                    r"already (?:written|contacted|raised|complained|reported)|contacted support about this before|"
                    r"keep raising|written \d+ times|more times than I can count|previous attempts|"
                    r"teesri baar|pehle bhi complaint|pehle bhi kai baar|kitni baar|multiple times", re.I)
FRAUD  = re.compile(r"did(?: not|n't) (?:make|do|take|withdraw|authori[sz]e)|do(?: not|n't) recognize|"
                    r"don't recall|do not recall|not mine|isn't mine|unauthori[sz]ed|unathorized|someone|"
                    r"kisi ne|maine nahi|maine koi payment nahi|anjaan|fraud|stolen|hacked|compromised|"
                    r"never submitted|not responsible|strange (?:cash|payment|withdrawal)|random (?:charge|withdrawal)", re.I)

# Updated AMT regex to require word boundaries specifically on 'k' unit suffixes to avoid false positives
AMT    = re.compile(r"(?:rs\.?|₹|inr)\s*([\d,]+(?:\.\d+)?)\s*\b(k|lakhs?|lac)?\b|"
                    r"([\d,]+(?:\.\d+)?)\s*\b(k|lakhs?|lac)\b|"
                    r"([\d,]+(?:\.\d+)?)\s*(?:rupees|rupaye)", re.I)

DISTRACT = re.compile(r"balance|limit|fee|pade hain|right now", re.I)
NP     = re.compile(r"N[\s.\-]*P[\s.\-]*((?:\d[\s.\-]*){9}\d)", re.I)
NDAYS  = re.compile(r"(\d+)\s*(?:days?|din)\s*(?:ago|pehle|pahle)", re.I)
REL    = [("day before yesterday",2),("parso",2),("last night",1),("yesterday",1),("kal",1),("today",0),("aaj",0)]
CHAN   = {"upi": r"\bupi\b|gpay|google pay|phonepe|bhim",
          "card": r"credit card|debit card|card payment|card transaction|\bvisa\b|mastercard|card se|card wala|by card|my card|card (?:was |got |is |keeps |being )?(?:declined|refused|rejected|denied)",
          "netbanking": r"net\s?-?\s?banking|internet banking", "wallet": r"wallet"}
HI     = re.compile(r"\b(hai|hain|nahi|nahin|karo|karna|karun|kaise|kyu|kyun|kya|mera|meri|mere|bhai|yaar|kal|"
                    r"parso|aaj|paise|paisa|rupaye|wapas|chahiye|liye|diya|gaya|gaye|hua|hui|raha|rahi|rahe|"
                    r"kitna|kitne|baar|din|pehle|nikal|maang|aur|kar|ho|jaldi|dhanyavaad|namaste|ji|ata|pata|"
                    r"chuka|hoon|tha|thi|kat|pade|abhi|lagega|bol)\b", re.I)

def body_of(u):
    b = u.split("\n\n",1)[1] if "\n\n" in u else u
    return "\n".join(l for l in b.splitlines() if not l.lstrip().startswith(">"))   # quoted emails don't count
def recv_of(u):
    m = re.search(r"Received:\s*(\d{4}-\d{2}-\d{2})", u); return date.fromisoformat(m.group(1)) if m else None
def sentences(t): return re.split(r"(?<=[.!?])\s+", t)
def amounts(t):
    """Return (transaction_amounts, distractor_amounts) in paise."""
    keep, drop = [], []
    for s in sentences(t):
        for m in AMT.finditer(s):
            num = m.group(1) or m.group(3) or m.group(5)
            unit = (m.group(2) or m.group(4) or "").lower()
            mult = 1000 if unit=="k" else 100000 if unit.startswith("lakh") or unit=="lac" else 1
            v = int(round(float(num.replace(",",""))*mult*100))
            (drop if DISTRACT.search(s) else keep).append(v)
    return keep, drop
def dates_expected(t, r):
    e=set()
    for w,d in REL:
        if re.search(r"\b"+w+r"\b", t, re.I): e.add((r-timedelta(days=d)).isoformat())
    for m in NDAYS.finditer(t): e.add((r-timedelta(days=int(m.group(1)))).isoformat())
    for m in re.finditer(r"\b\d{4}-\d{2}-\d{2}\b", t): e.add(m.group(0))
    for m in re.finditer(r"\b(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})\b", t):
        d,mo,y = int(m.group(1)), int(m.group(2)), int(m.group(3)); y += 2000 if y<100 else 0
        try: e.add(date(y,mo,d).isoformat())
        except ValueError: pass
    return e
def np_ids(t): return {"NP"+re.sub(r"\D","",m.group(1)) for m in NP.finditer(t)}
def norm_id(s):
    d = re.sub(r"\D","",s); return "NP"+d if len(d)==10 else None
def prio(cat, amt, legal):
    if cat=="fraud" or (amt is not None and amt>=5_000_000) or legal: return "P1"
    if (cat in ("payment_failed","refund") and amt is not None and amt>=500_000) or cat=="account_access": return "P2"
    return "P3"
def norm_user(u):  # for near-duplicate detection
    b = body_of(u).lower()
    b = re.sub(r"\bthx\b|[^\w\s]"," ",b); return re.sub(r"\s+"," ",b).strip()

def audit(row):
    """-> (drop_reason or None, fixes {field: (new, reason)}, flags [(field, why)])"""
    u = row["messages"][1]["content"]; raw = row["messages"][-1]["content"]
    try: lab = json.loads(raw); assert isinstance(lab, dict)
    except Exception: return "label JSON invalid/truncated", {}, []
    fixes, flags = {}, []
    if "prio" in lab and "priority" not in lab:                    # issue #2
        lab["priority"] = lab.pop("prio"); fixes["priority"] = (lab["priority"], "key 'prio'->'priority'")
    if set(lab) != set(KEYS): return f"key mismatch {set(lab)^set(KEYS)}", {}, []
    if type(lab["needs_human"]) is not bool: return "needs_human not bool", {}, []
    if lab["amount_paise"] is not None and type(lab["amount_paise"]) is not int: return "amount not int", {}, []
    if lab["category"] not in CATS: return "bad category", {}, []

    b = body_of(u); r = recv_of(u)
    if len(b.strip()) < 6 or b.strip().lower() in {"test","test test 123","asdf asdf"}:
        return "junk/empty ticket", {}, []

    # ---- deterministic fixes ----
    if isinstance(lab["txn_date"], str):                            # issue #3
        m = re.fullmatch(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", lab["txn_date"])
        if m:
            d,mo,y = int(m.group(1)),int(m.group(2)),int(m.group(3)); y += 2000 if y<100 else 0
            try:
                iso = date(y,mo,d).isoformat(); fixes["txn_date"]=(iso,"DD/MM/YYYY -> ISO"); lab["txn_date"]=iso
            except ValueError: flags.append(("txn_date","unparseable date"))
    if lab["txn_id"] is not None:                                   # issue #6
        nid = norm_id(lab["txn_id"])
        if nid and nid != lab["txn_id"]: fixes["txn_id"]=(nid,"normalize NP id"); lab["txn_id"]=nid
        elif nid is None:
            ids = np_ids(b)
            if len(ids)==1: fixes["txn_id"]=(next(iter(ids)),"id not NP+10digits; took from ticket"); lab["txn_id"]=next(iter(ids))
            else: fixes["txn_id"]=(None,"label id is not a NimbusPay id"); lab["txn_id"]=None
    if lab["category"] in NULL_CATS:                                # null rule
        for k in ("amount_paise","txn_date","txn_id","channel"):
            if lab[k] is not None: fixes[k]=(None,f"must be null for {lab['category']}"); lab[k]=None
    keep, dist = amounts(b)
    a = lab["amount_paise"]
    if len(keep)==1 and lab["category"] not in NULL_CATS:
        c = keep[0]
        if a is None: fixes["amount_paise"]=(c,"amount missing"); lab["amount_paise"]=c
        elif a!=c:
            for f,why in ((100,"rupees stored, x100"),(1000,"k not expanded"),(100000,"lakh not expanded")):
                if a*f==c: fixes["amount_paise"]=(c,why); lab["amount_paise"]=c; break
            else:
                if a==c*100: fixes["amount_paise"]=(c,"paise x100 too big"); lab["amount_paise"]=c
                else: flags.append(("amount_paise",f"label={a} ticket={c}"))
    elif len(keep)>1 and a not in keep: flags.append(("amount_paise",f"label={a} candidates={keep}"))
    p = prio(lab["category"], lab["amount_paise"], bool(LEGAL.search(b)))   # issue #7
    if lab["priority"] != p: fixes["priority"]=(p,f"rules give {p}"); lab["priority"]=p
    if (p=="P1" or REPEAT.search(b)) and lab["needs_human"] is not True:    # issue #8
        fixes["needs_human"]=(True,"P1 or repeat-contact"); lab["needs_human"]=True

    # ---- judgement calls: FLAG, never auto-fix ----
    if FRAUD.search(b) and lab["category"]!="fraud": flags.append(("category","fraud wording, label="+lab["category"]))
    if lab["needs_human"] is True and p!="P1" and not REPEAT.search(b): flags.append(("needs_human","true, no trigger found"))
    exp = dates_expected(b, r) if r else set()
    if lab["txn_date"] and exp and lab["txn_date"] not in exp: flags.append(("txn_date",f"label={lab['txn_date']} ticket implies {sorted(exp)}"))
    if lab["txn_date"] is None and exp and lab["category"] not in NULL_CATS: flags.append(("txn_date",f"null but ticket implies {sorted(exp)}"))
    if lab["txn_date"] and r and lab["txn_date"] > r.isoformat(): flags.append(("txn_date","after Received"))
    men = {c for c,pat in CHAN.items() if re.search(pat, b, re.I)}
    if lab["channel"] and lab["channel"] not in men: flags.append(("channel",f"{lab['channel']} not stated (found {men or 'none'})"))
    if lab["channel"] is None and len(men)==1 and lab["category"] not in NULL_CATS: flags.append(("channel",f"null but ticket says {men}"))
    hi = len({w.lower() for w in HI.findall(b)})
    if lab["language"]=="en" and hi>=2: flags.append(("language",f"en but {hi} Hinglish words"))
    if lab["language"]=="hi-en" and hi==0: flags.append(("language","hi-en but no Hinglish words"))
    return None, fixes, flags

In [ ]:
# 1) show EXACTLY what the auditor wants to change on clean dev rows
print("=== BEFORE PATCH ===")
for r in dev:
    drop, fixes, flags = audit(r)
    if drop or fixes or flags:
        print(r["id"], "| fixes:", fixes, "| flags:", flags)

# 2) patch the two known bugs
REPEAT = re.compile(r"third time|second time|second complaint|third complaint|again and again|baar baar|"
                    r"already (?:written|contacted|raised|complained|reported)|contacted support about this before|"
                    r"keep raising|written \d+ times|more times than I can count|previous attempts|"
                    r"teesri baar|pehle bhi complaint|pehle bhi kai baar|kitni baar", re.I)   # removed 'multiple times'

CHAN = {"upi": r"\bupi\b|\bgpay\b|google pay|phonepe|\bbhim\b",
        "card": r"credit card|debit card|\bvisa\b|mastercard|\bcards?\b",                      # broadened
        "netbanking": r"net\s?-?\s?banking|internet banking",
        "wallet": r"\bwallet\b"}

# 3) re-calibrate
print("\n=== AFTER PATCH ===")
bad = 0
for r in dev:
    drop, fixes, flags = audit(r)
    if drop or fixes or flags:
        bad += 1
        print(r["id"], "| fixes:", fixes, "| flags:", flags)
print(f"\nDEV rows auditor would change/flag: {bad} of {len(dev)}   <-- target 0-2")

In [ ]:
# 1) calibration
dev_flags = [(r["id"], audit(r)[2]) for r in dev if audit(r)[0] is None and (audit(r)[1] or audit(r)[2])]
print("DEV rows my auditor would change/flag:", len(dev_flags), "of", len(dev))
for i,f in dev_flags[:10]: print("  ", i, f)

# 2) clean train
seen, clean, log, review = {}, [], [], []
for r in train:
    rid, u = r["id"], r["messages"][1]["content"]
    if norm_user(u) in {norm_user(x["messages"][1]["content"]) for x in dev+test}:
        log.append((rid,"drop","leakage: same ticket as dev/test")); continue
    k = norm_user(u)
    if k in seen: log.append((rid,"drop",f"duplicate of {seen[k]}")); continue
    seen[k] = rid
    drop, fixes, flags = audit(r)
    if drop: log.append((rid,"drop",drop)); continue
    if fixes:
        lab = json.loads(r["messages"][-1]["content"])
        if "prio" in lab: lab["priority"] = lab.pop("prio")
        for f,(v,_) in fixes.items(): lab[f] = v
        r["messages"][-1]["content"] = json.dumps(lab, ensure_ascii=False)
        log.append((rid,"fix","; ".join(f"{f}: {why}" for f,(_,why) in fixes.items())))
    if flags: review.append({"id":rid,"flags":flags,"ticket":r["messages"][1]["content"],
                             "label":json.loads(r["messages"][-1]["content"])})
    clean.append(r)

with open(f"{SAVE}/cleaning_log.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["id","action","reason"]); w.writerows(log)
json.dump(review, open(f"{SAVE}/audit_flags.json","w"), indent=1, ensure_ascii=False)
with open(f"{SAVE}/train_clean.jsonl","w") as f:
    for r in clean: f.write(json.dumps(r,ensure_ascii=False)+"\n")

from collections import Counter
print(f"\nkept {len(clean)}/{len(train)} | log entries {len(log)} | need your eyes {len(review)}")
print("actions:", Counter(a for _,a,_ in log))
print("drop reasons:", Counter(x.split(':')[0] for _,a,x in log if a=="drop").most_common())
print("fix types:", Counter(f.split(':')[0] for _,a,x in log if a=="fix" for f in x.split('; ')).most_common())
print("flag types:", Counter(f for r in review for f,_ in r["flags"]).most_common())

In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = load_jsonl("pack/data/train_clean.jsonl")
dev = load_jsonl("pack/data/dev.jsonl")
test = load_jsonl("pack/data/test_inputs.jsonl")

print("Train:", len(train))
print("Dev:", len(dev))
print("Test:", len(test))
print(train[0])
df_train = pd.DataFrame(train)

print(df_train.shape)
print(df_train.columns.tolist())
df_train.head()

In [ ]:
import json
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Reload the data to fix the NameError
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = load_jsonl("pack/data/train_clean.jsonl")
dev = load_jsonl("pack/data/dev.jsonl")
test = load_jsonl("pack/data/test_inputs.jsonl")

df_train = pd.DataFrame(train)
df_dev = pd.DataFrame(dev)
df_test = pd.DataFrame(test)

# 2. Define the model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# 3. Convert pandas DataFrames to HuggingFace Datasets
train_dataset = Dataset.from_pandas(df_train)
dev_dataset = Dataset.from_pandas(df_dev)

# 4. Function to calculate actual token length using Qwen's chat template
def get_token_length(messages):
    # tokenize=False gives us the raw string formatted exactly how Qwen reads it
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return len(tokenizer(text)["input_ids"])

# Apply to check lengths
df_train['token_length'] = df_train['messages'].apply(get_token_length)

# 5. Output Statistics
print("=== Token Length Statistics ===")
print(df_train['token_length'].describe())
print(f"\nMax Sequence Length: {df_train['token_length'].max()} tokens")

In [ ]:
import torch, os, json
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from google.colab import drive

# Mount Drive so we don't lose the adapter if Colab disconnects!
drive.mount('/content/drive', force_remount=True)
SAVE_DIR = "/content/drive/MyDrive/nimbuspay_3b"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Reload data
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

df_train = pd.DataFrame(load_jsonl("pack/data/train_clean.jsonl"))

# 2. Configure model (Using Unsloth's pre-quantized 3B for faster loading and less VRAM)
max_seq_length = 2048
model_id = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" # 3B is the largest Qwen <= 4B

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# 3. Format Data
def formatting_prompts_func(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = Dataset.from_pandas(df_train).map(formatting_prompts_func)

# 4. Training Arguments (FIXED: warmup_ratio -> warmup_steps)
training_args = SFTConfig(
    output_dir=f"{SAVE_DIR}/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=10,  # <--- FIXED (trl removed warmup_ratio)
    max_seq_length=max_seq_length,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

# 5. Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

print("🚀 Starting Qwen2.5-3B Fine-Tuning... (Will take ~20-25 mins on T4)")
trainer.train()

# 6. Save to Drive
FINAL_DIR = f"{SAVE_DIR}/final_adapter"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✅ Training complete and saved to {FINAL_DIR}!")

In [5]:
import os, json, importlib.util, torch
from unsloth import FastLanguageModel
from peft import PeftModel

# 1. Paths
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
ADAPTER_PATH = "/content/drive/MyDrive/nimbuspay_3b/final_adapter"

# 2. Load Base + Adapter (max_seq_length=4096 to prevent crash on long emails!)
print("Loading model and adapter...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
FastLanguageModel.for_inference(model)
print("✅ 3B Model and adapter loaded successfully!")

# 3. Inference Loop on Dev Set
dev_data = [json.loads(l) for l in open("pack/data/dev.jsonl", "r", encoding="utf-8") if l.strip()]
preds = []
print("Generating predictions for Dev Set...")
for i, row in enumerate(dev_data):
    messages = row["messages"][:-1] # Drop the gold answer
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # Strict assignment rules: Greedy decoding, max 200 tokens
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    preds.append({"id": row["id"], "output": response})

# Save dev predictions locally
with open("dev_preds_3b.jsonl", "w") as f:
    for p in preds: f.write(json.dumps(p) + "\n")

# 4. Score it using the provided score.py
spec = importlib.util.spec_from_file_location("scorer", "pack/score.py")
scorer = importlib.util.module_from_spec(spec); spec.loader.exec_module(scorer)
res = scorer.score(dev_data, preds)

print(f"\n🎯 3B Dev Exact Match: {res['all']['exact_match']:.2%}")
print(f"🎯 3B Dev Mean Field Acc: {res['all']['mean_field_acc']:.2%}")

Loading model and adapter...
==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.15.0+cu132. CUDA: 7.5. CUDA Toolkit: 13.2. Triton: 3.9.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ 3B Model and adapter loaded successfully!
Generating predictions for Dev Set...


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


🎯 3B Dev Exact Match: 94.00%
🎯 3B Dev Mean Field Acc: 98.88%


In [6]:
# 1. Load test data
test_data = [json.loads(l) for l in open("pack/data/test_inputs.jsonl", "r", encoding="utf-8") if l.strip()]
test_preds = []

print(f"Generating predictions for {len(test_data)} test tickets... (This will take ~10-15 mins)")
for i, row in enumerate(test_data):
    messages = row["messages"] # Only contains system and user
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    test_preds.append({"id": row["id"], "output": response})

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(test_data)}...")

# 2. Save Predictions safely to Drive and locally
import shutil
os.makedirs("submission", exist_ok=True)
with open("submission/predictions_test.jsonl", "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# Also save to your Drive so you don't lose it if Colab disconnects!
SAVE_DIR = "/content/drive/MyDrive/nimbuspay_3b"
shutil.copy("submission/predictions_test.jsonl", f"{SAVE_DIR}/predictions_test.jsonl")

print(f"\n🎉 SUCCESS! Test predictions saved to:")
print(f"   1. Local: ./submission/predictions_test.jsonl")
print(f"   2. Drive: {SAVE_DIR}/predictions_test.jsonl")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating predictions for 400 test tickets... (This will take ~10-15 mins)


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Processed 50/400...


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Processed 100/400...


... (8 outputs left out)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Processed 350/400...


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Processed 400/400...

🎉 SUCCESS! Test predictions saved to:
   1. Local: ./submission/predictions_test.jsonl
   2. Drive: /content/drive/MyDrive/nimbuspay_3b/predictions_test.jsonl


In [7]:
# Load the dev predictions we just generated
dev_preds_dict = {p["id"]: p["output"] for p in preds}
failures = []

for row in dev_data:
    gold = json.loads(row["messages"][-1]["content"])
    try:
        pred = json.loads(dev_preds_dict[row["id"]])
        # The grader checks exact match and types
        if pred != gold:
            failures.append({
                "id": row["id"],
                "ticket_snippet": row["messages"][1]["content"][:250] + "...",
                "gold": gold,
                "pred": pred
            })
    except json.JSONDecodeError:
        failures.append({"id": row["id"], "error": "Model output was not valid JSON", "ticket_snippet": row["messages"][1]["content"][:250]})

print(f"Found exactly {len(failures)} failures on the dev set.")
print("\n--- PASTE THESE INTO YOUR report.md (Part E) ---")
import pprint
# Print the first 10 failures
pprint.pprint(failures[:10], width=120)

Found exactly 12 failures on the dev set.

--- PASTE THESE INTO YOUR report.md (Part E) ---
[{'gold': {'amount_paise': 13000000,
           'category': 'fraud',
           'channel': None,
           'language': 'en',
           'needs_human': True,
           'priority': 'P1',
           'txn_date': '2026-06-30',
           'txn_id': 'NP7545379360'},
  'id': 'dv-00006',
  'pred': {'amount_paise': 13000000,
           'category': 'fraud',
           'channel': None,
           'language': 'en',
           'needs_human': True,
           'priority': 'P1',
           'txn_date': '2026-07-01',
           'txn_id': 'NP7545379360'},
  'ticket_snippet': 'Received: 2026-07-02 (Thu)\n'
                    'Source: twitter\n'
                    '\n'
                    '@NimbusPayCare The App shows a fraudulent payment made on my account. It was day before '
                    'yesterday. You can check np 75453 79360. Amount: 1.3 lakh...'},
 {'gold': {'amount_paise': 3673700,
           'cate